# Cox Regression Penalized

### Imports

In [1]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree
from sklearn.model_selection import train_test_split
from sksurv.ensemble import RandomSurvivalForest
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.linear_model import CoxPHSurvivalAnalysis
from sksurv.metrics import concordance_index_censored , concordance_index_ipcw
from sklearn.impute import SimpleImputer
from sksurv.util import Surv

# Clinical Data
df = pd.read_csv("./data/X_train/clinical_train.csv")
df_eval = pd.read_csv("./data/X_test/clinical_test.csv")

# Molecular Data
maf_df = pd.read_csv("./data/X_train/molecular_train.csv")
maf_eval = pd.read_csv("./data/X_test/molecular_test.csv")

target_df = pd.read_csv("./data/target_train.csv")

# TODO : Uncomment for test data ??
"""
target_df_test = pd.read_csv("./data/target_test.csv")
"""
# Preview the data
df.head()

,ID,CENTER,BM_BLAST,WBC,ANC,MONOCYTES,HB,PLT,CYTOGENETICS
0,P132697,MSK,14.0,2.8,0.2,0.7,7.6,119.0,"46,xy,del(20)(q12)[2]/46,xy[18]"
1,P132698,MSK,1.0,7.4,2.4,0.1,11.6,42.0,"46,xx"
2,P116889,MSK,15.0,3.7,2.1,0.1,14.2,81.0,"46,xy,t(3;3)(q25;q27)[8]/46,xy[12]"
3,P132699,MSK,1.0,3.9,1.9,0.1,8.9,77.0,"46,xy,del(3)(q26q27)[15]/46,xy[5]"
4,P132700,MSK,6.0,128.0,9.7,0.9,11.1,195.0,"46,xx,t(3;9)(p13;q22)[10]/46,xx[10]"


### Cleaning

In [2]:
# Drop rows where 'OS_YEARS' is NaN if conversion caused any issues
target_df.dropna(subset=['OS_YEARS', 'OS_STATUS'], inplace=True)

# Check the data types to ensure 'OS_STATUS' is boolean and 'OS_YEARS' is numeric
print(target_df[['OS_STATUS', 'OS_YEARS']].dtypes)

# Contarget_dfvert 'OS_YEARS' to numeric if it isn’t already
target_df['OS_YEARS'] = pd.to_numeric(target_df['OS_YEARS'], errors='coerce')

# Ensure 'OS_STATUS' is boolean
target_df['OS_STATUS'] = target_df['OS_STATUS'].astype(bool)

OS_STATUS    float64
OS_YEARS     float64
dtype: object


1. Extracting number of somatic mutations 

In [3]:
# Step: Extract the number of somatic mutations per patient
# Group by 'ID' and count the number of mutations (rows) per patient
tmp = maf_df.groupby('ID').size().reset_index(name='Nmut')

# Merge with the training dataset and replace missing values in 'Nmut' with 0
df_2 = df.merge(tmp, on='ID', how='left').fillna({'Nmut': 0})

2. One hot encoding 'GENE', 'EFFECT' and 2 first relevant characters of 'PROTEIN_CHANGE'

In [4]:
def parse_protein(protein):
    protein = str(protein)

    if len(protein) == 0 or len(protein) == 1:
        return pd.NA
    
    try:
        protein = str(protein.split('.')[1])
    
    except IndexError:
        return pd.NA
    
    if len(protein) < 2 and protein != '?':
        print("protein with length <2:", protein)
        return protein
    
    else:
        return protein[0:2]

In [5]:
# One-hot encode the 'GENE', 'EFFECT' and 'PROTEIN_CHANGE' columns
one_hot_encoded_genes = pd.get_dummies(maf_df['GENE'], prefix='GENE')
one_hot_encoded_effects = pd.get_dummies(maf_df['EFFECT'], prefix='EFFECT')
one_hot_encoded_proteins = pd.get_dummies(maf_df['PROTEIN_CHANGE'].apply(parse_protein), prefix='PROTEIN')

# Add the one-hot encoded columns to the original DataFrame
maf_df_one_hot_genes = pd.concat([maf_df[['ID']], one_hot_encoded_genes], axis=1)
maf_df_one_hot_effects = pd.concat([maf_df[['ID']], one_hot_encoded_effects], axis=1)
maf_df_one_hot_proteins = pd.concat([maf_df[['ID']], one_hot_encoded_proteins], axis=1)

# Group by 'ID' and aggregate using max to ensure one entry per ID
maf_df_one_hot_genes = maf_df_one_hot_genes.groupby('ID').max().reset_index()
maf_df_one_hot_effects = maf_df_one_hot_effects.groupby('ID').max().reset_index()
maf_df_one_hot_proteins = maf_df_one_hot_proteins.groupby('ID').max().reset_index()

# Merge the one-hot encoded DataFrames
maf_df_one_hot_grouped = maf_df_one_hot_proteins.merge(maf_df_one_hot_genes.merge(maf_df_one_hot_effects, on='ID', how='outer'), on='ID', how='outer').fillna(0)
maf_df_one_hot_grouped.head()

,ID,PROTEIN_*3,PROTEIN_*6,PROTEIN_?,PROTEIN_A1,PROTEIN_A2,PROTEIN_A3,PROTEIN_A4,PROTEIN_A5,PROTEIN_A6,...,EFFECT_inframe_codon_gain,EFFECT_inframe_codon_loss,EFFECT_inframe_variant,EFFECT_initiator_codon_change,EFFECT_non_synonymous_codon,EFFECT_splice_site_variant,EFFECT_stop_gained,EFFECT_stop_lost,EFFECT_stop_retained_variant,EFFECT_synonymous_codon
0,P100000,False,False,True,False,False,False,False,False,False,...,False,False,False,False,True,True,True,False,False,False
1,P100001,False,False,False,False,False,False,False,False,False,...,False,False,False,False,True,False,True,False,False,False
2,P100002,False,False,False,False,False,False,False,False,False,...,False,False,False,False,True,False,True,False,False,False
3,P100004,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,P100006,False,False,False,False,False,False,False,False,False,...,False,False,False,False,True,False,True,False,False,False


### 3. Training

1. Selecting features and getting rid of rows with NaN values in training set

In [6]:
# Select features and target columns
features = ['BM_BLAST', 'HB', 'PLT', 'Nmut', 'ANC','WBC']
one_hot_features_genes = list(maf_df_one_hot_genes.columns[1:])
one_hot_features_effects = list(maf_df_one_hot_effects.columns[1:])
one_hot_encoded_proteins = list(maf_df_one_hot_proteins.columns[1:])

# Final feature list
total_features = features + one_hot_features_effects + one_hot_features_genes + one_hot_encoded_proteins

target = ['OS_YEARS', 'OS_STATUS']

# Merge on ID to ensure same row order
merged = df_2[['ID','CENTER'] + features].merge(target_df[['ID'] + target], on='ID', how='left')
merged = merged.merge(maf_df_one_hot_grouped, on='ID', how='left')  # Merge one-hot encoded genes

# Remove rows with missing feature values
merged = merged.dropna()

# Build feature matrix and survival target
X = merged[total_features]
y = Surv.from_dataframe('OS_STATUS', 'OS_YEARS', merged[target])

2. Spliting dataset

In [7]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
X_train.head()

,BM_BLAST,HB,PLT,Nmut,ANC,WBC,EFFECT_2KB_upstream_variant,EFFECT_3_prime_UTR_variant,EFFECT_ITD,EFFECT_PTD,...,PROTEIN_W9,PROTEIN_Y1,PROTEIN_Y2,PROTEIN_Y3,PROTEIN_Y4,PROTEIN_Y5,PROTEIN_Y6,PROTEIN_Y7,PROTEIN_Y8,PROTEIN_Y9
1793,5.0,10.600,175.0,5.0,2.80,3.8,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3043,3.0,9.100,247.0,2.0,1.90,4.9,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2920,1.0,6.927,274.0,3.0,2.00,4.2,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
918,2.0,6.600,51.0,5.0,5.06,8.3,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2923,18.0,9.022,103.0,6.0,0.70,1.9,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


3. Fitting Cox Ridge model

In [32]:
# Initialize and train the Cox Proportional Hazards model
cox = CoxnetSurvivalAnalysis(l1_ratio=1, alpha_min_ratio=0.001, max_iter=10000, normalize=True)
cox.fit(X_train, y_train)

# Evaluate the model using Concordance Index IPCW
cox_cindex_train = concordance_index_ipcw(y_train, y_train, cox.predict(X_train), tau=7)[0]
cox_cindex_test = concordance_index_ipcw(y_train, y_test, cox.predict(X_test), tau=7)[0]
print(f"Cox Proportional Hazard Model Concordance Index IPCW on train: {cox_cindex_train:.5f}")
print(f"Cox Proportional Hazard Model Concordance Index IPCW on test: {cox_cindex_test:.5f}")

Cox Proportional Hazard Model Concordance Index IPCW on train: 0.73174
Cox Proportional Hazard Model Concordance Index IPCW on test: 0.70665


/tmp/ipykernel_76234/2955706528.py:3: ConvergenceWarning: Optimization terminated early, you might want to increase the number of iterations (max_iter=10000).
  cox.fit(X_train, y_train)


### 4. Prediction

1. Calculating Nmut on prediction sample

In [33]:
tmp_eval = maf_eval.groupby('ID').size().reset_index(name='Nmut')

# Merge with the training dataset and replace missing values in 'Nmut' with 0
df_eval2 = df_eval.merge(tmp_eval, on='ID', how='left').fillna({'Nmut': 0})

2. One hot encoding 'GENE' and 'EFFECT'

In [34]:
# One-hot encode the 'GENE' and 'EFFECT' column
one_hot_encoded_genes_eval = pd.get_dummies(maf_eval['GENE'], prefix='GENE')
one_hot_encoded_effects_eval = pd.get_dummies(maf_eval['EFFECT'], prefix='EFFECT')
one_hot_encoded_proteins_eval = pd.get_dummies(maf_eval['PROTEIN_CHANGE'].apply(parse_protein), prefix='PROTEIN')

# Add the one-hot encoded columns to the original DataFrame
maf_eval_one_hot_genes = pd.concat([maf_eval[['ID']], one_hot_encoded_genes_eval], axis=1)
maf_eval_one_hot_effects = pd.concat([maf_eval[['ID']], one_hot_encoded_effects_eval], axis=1)
maf_eval_one_hot_proteins = pd.concat([maf_eval[['ID']], one_hot_encoded_proteins_eval], axis=1)

# Group by 'ID' and aggregate using max to ensure one entry per ID
maf_eval_one_hot_grouped = maf_eval_one_hot_proteins.merge(maf_eval_one_hot_genes.merge(maf_eval_one_hot_effects, on='ID', how='outer'), on='ID', how='outer').fillna(0)
maf_eval_one_hot_grouped = maf_eval_one_hot_grouped.groupby('ID').max().reset_index()

maf_eval_one_hot_grouped.head()

,ID,PROTEIN_10,PROTEIN_11,PROTEIN_12,PROTEIN_13,PROTEIN_14,PROTEIN_15,PROTEIN_16,PROTEIN_18,PROTEIN_19,...,GENE_WT1,GENE_ZRSR2,EFFECT_ITD,EFFECT_PTD,EFFECT_frameshift_variant,EFFECT_inframe_codon_gain,EFFECT_inframe_codon_loss,EFFECT_non_synonymous_codon,EFFECT_stop_gained,EFFECT_stop_lost
0,KYW1,False,False,False,False,False,False,False,False,False,...,False,False,True,False,True,False,False,True,True,False
1,KYW10,False,False,False,False,False,False,False,False,False,...,False,True,False,False,False,False,False,True,True,False
2,KYW100,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,True,False
3,KYW1000,False,False,False,False,False,False,False,False,False,...,False,True,False,False,True,False,False,False,True,False
4,KYW1001,False,False,False,False,False,False,False,False,False,...,False,True,False,False,True,False,False,True,True,False


3. Align colums of X_eval with those of X_train

In [35]:
X_eval = df_eval2[['ID'] + features].merge(maf_eval_one_hot_grouped, on='ID', how='left')
X_eval = X_eval.drop(columns=['ID'])

# Ensure X is a pandas DataFrame before accessing its columns
if not isinstance(X, pd.DataFrame):
	X = pd.DataFrame(X, columns=total_features)

X_eval = X_eval.reindex(columns=X.columns, fill_value=False)
X_eval = X_eval.drop(columns=[col for col in X_eval.columns if col not in total_features])
X_eval.head()

,BM_BLAST,HB,PLT,Nmut,ANC,WBC,EFFECT_2KB_upstream_variant,EFFECT_3_prime_UTR_variant,EFFECT_ITD,EFFECT_PTD,...,PROTEIN_W9,PROTEIN_Y1,PROTEIN_Y2,PROTEIN_Y3,PROTEIN_Y4,PROTEIN_Y5,PROTEIN_Y6,PROTEIN_Y7,PROTEIN_Y8,PROTEIN_Y9
0,68.0,7.6,48.0,4.0,0.5865,3.45,False,False,True,False,...,False,False,False,False,False,False,False,False,False,False
1,35.0,10.0,32.0,3.0,1.2402,3.18,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,NaN,12.3,25.0,3.0,8.6800,12.40,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,61.0,8.0,44.0,3.0,2.0535,5.55,False,False,True,False,...,False,False,False,False,False,False,False,False,False,False
4,2.0,8.6,27.0,3.0,0.7381,1.21,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


3. Imputer with median strategy

In [63]:
imputer = SimpleImputer(strategy='median')
print(features)
X_train[features] = imputer.fit_transform(X_train[features])
X_eval[features] = imputer.transform(X_eval[features])
X_eval = X_eval.fillna(False)
X_eval.head()

['BM_BLAST', 'HB', 'PLT', 'Nmut', 'ANC', 'WBC']


/tmp/ipykernel_71081/926956949.py:5: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_eval = X_eval.fillna(False)


,BM_BLAST,HB,PLT,Nmut,ANC,WBC,EFFECT_2KB_upstream_variant,EFFECT_3_prime_UTR_variant,EFFECT_ITD,EFFECT_PTD,...,PROTEIN_W9,PROTEIN_Y1,PROTEIN_Y2,PROTEIN_Y3,PROTEIN_Y4,PROTEIN_Y5,PROTEIN_Y6,PROTEIN_Y7,PROTEIN_Y8,PROTEIN_Y9
0,68.0,7.6,48.0,4.0,0.5865,3.45,False,False,True,False,...,False,False,False,False,False,False,False,False,False,False
1,35.0,10.0,32.0,3.0,1.2402,3.18,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,3.5,12.3,25.0,3.0,8.6800,12.40,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,61.0,8.0,44.0,3.0,2.0535,5.55,False,False,True,False,...,False,False,False,False,False,False,False,False,False,False
4,2.0,8.6,27.0,3.0,0.7381,1.21,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [64]:
prediction_on_test_set = cox.predict(X_eval)
prediction_on_test_set

array([ 2.03263033,  0.78874716, -0.26493029, ...,  0.46214168,
       -0.31158473,  0.53475823])

In [65]:
submission = pd.Series(prediction_on_test_set, index=df_eval['ID'], name='risk_score')

In [66]:
submission.to_csv('./submission/cox_ridge_submission.csv')

In [58]:
submission.head()

ID
KYW1    2.032630
KYW2    0.779324
KYW3   -0.354906
KYW4    1.886326
KYW5    0.155011
Name: risk_score, dtype: float64